In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Makes our synthetic data reproducible
np.random.seed(42)

N_LEADS = 20000

START_DATE = pd.Timestamp("2024-01-01")
END_DATE = pd.Timestamp("2025-12-31")

In [3]:
programmes = pd.DataFrame({
    "programme_id": ["P001", "P002", "P003", "P004", "P005"],
    "programme_name": [
        "Data Analytics",
        "Data Science",
        "Business Analytics",
        "Cybersecurity",
        "Digital Marketing"
    ],
    "category": [
        "Data",
        "Data",
        "Business",
        "Technology",
        "Marketing"
    ],
    "price": [499, 599, 449, 549, 399],
    "duration_weeks": [12, 16, 12, 16, 10]
})

programmes

,programme_id,programme_name,category,price,duration_weeks
0,P001,Data Analytics,Data,499,12
1,P002,Data Science,Data,599,16
2,P003,Business Analytics,Business,449,12
3,P004,Cybersecurity,Technology,549,16
4,P005,Digital Marketing,Marketing,399,10


In [5]:
channels = ["LinkedIn", "Google Ads", "Meta Ads", "University Partnership", "Referral"]

campaigns = []

for i in range(1, 41):
    channel = np.random.choice(channels)
    
    start_date = START_DATE + pd.Timedelta(
        days=np.random.randint(0, (END_DATE - START_DATE).days - 30)
    )
    
    duration = np.random.randint(14, 61)
    end_date = min(start_date + pd.Timedelta(days=duration), END_DATE)

    spend = round(np.random.uniform(500, 5000), 2)

    impressions = np.random.randint(10000, 150000)
    clicks = np.random.randint(500, min(15000, impressions))

    campaigns.append([
        f"C{i:03d}",
        f"{channel} Campaign {i}",
        channel,
        start_date,
        end_date,
        spend,
        impressions,
        clicks
    ])

campaigns = pd.DataFrame(campaigns, columns=[
    "campaign_id",
    "campaign_name",
    "channel",
    "start_date",
    "end_date",
    "campaign_spend",
    "impressions",
    "clicks"
])

campaigns.head()

,campaign_id,campaign_name,channel,start_date,end_date,campaign_spend,impressions,clicks
0,C001,University Partnership Campaign 1,University Partnership,2024-09-27,2024-11-22,4008.61,64886,6765
1,C002,Meta Ads Campaign 2,Meta Ads,2024-08-02,2024-08-26,2566.62,136324,11863
2,C003,Meta Ads Campaign 3,Meta Ads,2025-10-23,2025-11-07,3748.99,74925,7449
3,C004,Google Ads Campaign 4,Google Ads,2024-07-10,2024-08-13,3278.67,95305,12809
4,C005,Referral Campaign 5,Referral,2025-07-14,2025-08-23,3253.34,132537,2547


In [7]:
# Generate random lead creation dates
date_range_days = (END_DATE - START_DATE).days

created_dates = START_DATE + pd.to_timedelta(
    np.random.randint(0, date_range_days + 1, N_LEADS),
    unit="D"
)

leads = pd.DataFrame({
    "lead_id": [f"L{i:05d}" for i in range(1, N_LEADS + 1)],
    "created_date": created_dates,
    "age_group": np.random.choice(
        ["18-21", "22-25", "26-30", "31+"],
        N_LEADS,
        p=[0.35, 0.40, 0.18, 0.07]
    ),
    "education_level": np.random.choice(
        ["Undergraduate", "Postgraduate", "Graduate", "Other"],
        N_LEADS,
        p=[0.40, 0.30, 0.25, 0.05]
    ),
    "employment_status": np.random.choice(
        ["Student", "Employed", "Unemployed"],
        N_LEADS,
        p=[0.55, 0.25, 0.20]
    ),
    "country": np.random.choice(
        ["UK", "India", "USA", "Canada", "Other"],
        N_LEADS,
        p=[0.35, 0.30, 0.12, 0.08, 0.15]
    ),
    "acquisition_channel": np.random.choice(
        channels,
        N_LEADS,
        p=[0.30, 0.20, 0.18, 0.12, 0.20]
    ),
    "programme_interest": np.random.choice(
        programmes["programme_id"],
        N_LEADS
    )
})

leads.head()

,lead_id,created_date,age_group,education_level,employment_status,country,acquisition_channel,programme_interest
0,L00001,2024-01-09,22-25,Undergraduate,Student,India,Referral,P004
1,L00002,2024-08-20,22-25,Graduate,Student,UK,Meta Ads,P001
2,L00003,2024-04-08,22-25,Postgraduate,Student,UK,LinkedIn,P005
3,L00004,2025-10-20,22-25,Graduate,Student,UK,LinkedIn,P005
4,L00005,2024-07-26,22-25,Undergraduate,Student,Canada,LinkedIn,P001


In [9]:
leads["campaign_id"] = None

for channel in ["LinkedIn", "Google Ads", "Meta Ads"]:
    
    available_campaigns = campaigns.loc[
        campaigns["channel"] == channel,
        "campaign_id"
    ].values
    
    mask = leads["acquisition_channel"] == channel
    
    leads.loc[mask, "campaign_id"] = np.random.choice(
        available_campaigns,
        size=mask.sum()
    )

leads.head(10)

,lead_id,created_date,age_group,education_level,employment_status,country,acquisition_channel,programme_interest,campaign_id
0,L00001,2024-01-09,22-25,Undergraduate,Student,India,Referral,P004,None
1,L00002,2024-08-20,22-25,Graduate,Student,UK,Meta Ads,P001,C024
2,L00003,2024-04-08,22-25,Postgraduate,Student,UK,LinkedIn,P005,C015
3,L00004,2025-10-20,22-25,Graduate,Student,UK,LinkedIn,P005,C030
4,L00005,2024-07-26,22-25,Undergraduate,Student,Canada,LinkedIn,P001,C008
5,L00006,2024-05-10,18-21,Postgraduate,Student,UK,LinkedIn,P003,C030
6,L00007,2025-02-07,26-30,Undergraduate,Student,UK,University Partnership,P003,None
7,L00008,2024-05-31,26-30,Undergraduate,Unemployed,UK,Google Ads,P004,C011
8,L00009,2024-02-23,18-21,Postgraduate,Student,India,Referral,P005,None
9,L00010,2024-04-29,26-30,Graduate,Employed,UK,Google Ads,P001,C014


In [11]:
interaction_types = ["Call", "Email", "Brochure Sent", "Webinar"]
outcomes = ["Interested", "No Response", "Follow-up Required", "Not Interested"]

interactions = []
interaction_id = 1

for _, lead in leads.iterrows():

    # Each lead gets 0–4 interactions
    n = np.random.choice(
        [0, 1, 2, 3, 4],
        p=[0.15, 0.30, 0.30, 0.15, 0.10]
    )

    for _ in range(n):

        interaction_date = lead["created_date"] + pd.Timedelta(
            days=np.random.randint(0, 31)
        )

        interactions.append([
            f"I{interaction_id:06d}",
            lead["lead_id"],
            interaction_date,
            np.random.choice(interaction_types),
            np.random.choice(["Yes", "No"], p=[0.75, 0.25]),
            np.random.choice(outcomes)
        ])

        interaction_id += 1

interactions = pd.DataFrame(interactions, columns=[
    "interaction_id",
    "lead_id",
    "interaction_date",
    "interaction_type",
    "staff_contacted",
    "outcome"
])

interactions.head()

,interaction_id,lead_id,interaction_date,interaction_type,staff_contacted,outcome
0,I000001,L00001,2024-01-15,Call,Yes,Not Interested
1,I000002,L00002,2024-08-23,Webinar,Yes,No Response
2,I000003,L00002,2024-09-11,Brochure Sent,Yes,Follow-up Required
3,I000004,L00002,2024-08-29,Brochure Sent,Yes,No Response
4,I000005,L00002,2024-09-04,Webinar,Yes,Interested


In [13]:
interactions.shape

(35249, 6)

In [15]:
# Different channels have different base conversion probabilities
conversion_rates = {
    "LinkedIn": 0.07,
    "Google Ads": 0.06,
    "Meta Ads": 0.05,
    "University Partnership": 0.12,
    "Referral": 0.15
}

enrolments = []

for _, lead in leads.iterrows():

    probability = conversion_rates[lead["acquisition_channel"]]

    if np.random.random() < probability:

        enrolment_date = lead["created_date"] + pd.Timedelta(
            days=np.random.randint(1, 46)
        )

        enrolments.append([
            f"E{len(enrolments)+1:05d}",
            lead["lead_id"],
            lead["programme_interest"],
            enrolment_date,
            np.random.choice(
                ["Completed", "In Progress", "Dropped Out"],
                p=[0.65, 0.20, 0.15]
            )
        ])

enrolments = pd.DataFrame(enrolments, columns=[
    "enrolment_id",
    "lead_id",
    "programme_id",
    "enrolment_date",
    "status"
])

enrolments.head()

,enrolment_id,lead_id,programme_id,enrolment_date,status
0,E00001,L00009,P005,2024-02-27,Completed
1,E00002,L00034,P005,2025-06-01,Dropped Out
2,E00003,L00038,P002,2025-02-10,In Progress
3,E00004,L00048,P005,2024-09-07,In Progress
4,E00005,L00049,P003,2025-02-03,Completed


In [17]:
enrolments.shape

(1712, 5)

In [19]:
enrolments["completion_date"] = pd.NaT

for idx, row in enrolments.iterrows():
    if row["status"] == "Completed":
        
        duration = programmes.loc[
            programmes["programme_id"] == row["programme_id"],
            "duration_weeks"
        ].iloc[0]
        
        enrolments.loc[idx, "completion_date"] = (
            row["enrolment_date"] + pd.Timedelta(weeks=int(duration))
        )

enrolments.head(10)

,enrolment_id,lead_id,programme_id,enrolment_date,status,completion_date
0,E00001,L00009,P005,2024-02-27,Completed,2024-05-07
1,E00002,L00034,P005,2025-06-01,Dropped Out,NaT
2,E00003,L00038,P002,2025-02-10,In Progress,NaT
3,E00004,L00048,P005,2024-09-07,In Progress,NaT
4,E00005,L00049,P003,2025-02-03,Completed,2025-04-28
5,E00006,L00057,P003,2024-11-12,Dropped Out,NaT
6,E00007,L00071,P003,2024-02-17,Completed,2024-05-11
7,E00008,L00078,P004,2025-05-26,Completed,2025-09-15
8,E00009,L00082,P001,2024-05-02,Completed,2024-07-25
9,E00010,L00096,P001,2025-10-27,In Progress,NaT


In [21]:
payments = []

for _, enrolment in enrolments.iterrows():

    price = programmes.loc[
        programmes["programme_id"] == enrolment["programme_id"],
        "price"
    ].iloc[0]

    payment_date = enrolment["enrolment_date"]

    payment_status = np.random.choice(
        ["Successful", "Failed", "Refunded"],
        p=[0.90, 0.04, 0.06]
    )

    refund_amount = price if payment_status == "Refunded" else 0

    payments.append([
        f"PAY{len(payments)+1:05d}",
        enrolment["enrolment_id"],
        payment_date,
        price,
        np.random.choice(["Card", "Bank Transfer", "PayPal"]),
        payment_status,
        refund_amount
    ])

payments = pd.DataFrame(payments, columns=[
    "payment_id",
    "enrolment_id",
    "payment_date",
    "amount",
    "payment_method",
    "payment_status",
    "refund_amount"
])

payments.head()

,payment_id,enrolment_id,payment_date,amount,payment_method,payment_status,refund_amount
0,PAY00001,E00001,2024-02-27,399,PayPal,Successful,0
1,PAY00002,E00002,2025-06-01,399,Card,Successful,0
2,PAY00003,E00003,2025-02-10,599,Card,Successful,0
3,PAY00004,E00004,2024-09-07,399,Card,Successful,0
4,PAY00005,E00005,2025-02-03,449,Bank Transfer,Successful,0


In [23]:
programmes.to_csv("programmes.csv", index=False)
campaigns.to_csv("campaigns.csv", index=False)
leads.to_csv("leads.csv", index=False)
interactions.to_csv("interactions.csv", index=False)
enrolments.to_csv("enrolments.csv", index=False)
payments.to_csv("payments.csv", index=False)

print("All datasets exported successfully.")

All datasets exported successfully.


In [25]:
print("Programmes:", programmes.shape)
print("Campaigns:", campaigns.shape)
print("Leads:", leads.shape)
print("Interactions:", interactions.shape)
print("Enrolments:", enrolments.shape)
print("Payments:", payments.shape)

print("\nMissing IDs:")
print("Lead IDs:", leads["lead_id"].isna().sum())
print("Interaction Lead IDs:", interactions["lead_id"].isna().sum())
print("Enrolment Lead IDs:", enrolments["lead_id"].isna().sum())
print("Payment Enrolment IDs:", payments["enrolment_id"].isna().sum())

Programmes: (5, 5)
Campaigns: (40, 8)
Leads: (20000, 9)
Interactions: (35249, 6)
Enrolments: (1712, 6)
Payments: (1712, 7)

Missing IDs:
Lead IDs: 0
Interaction Lead IDs: 0
Enrolment Lead IDs: 0
Payment Enrolment IDs: 0


In [27]:
import os
print(os.getcwd())

/Users/tanayajadhav
